<a href="https://colab.research.google.com/github/bijelic02/myRAGProject/blob/main/MyRAGProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase **1**
Environment & Data Setup



Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Mounted successfully")

This mounts your Google Drive into Colab. When you run it, it asks for permission once and then your Drive is accessible at `/content/drive/MyDrive/`. Your notebook file itself (`.ipynb`) gets saved to Drive automatically if you set it up right.

**Save the notebook itself to Drive**

In Colab go to:
```
File → Save a copy in Drive
```

This moves your notebook from Colab's temporary storage into your actual Google Drive. From that point on, `Ctrl+S` saves it there. Never work from the default "Colab Notebooks" auto-created file without doing this first.
---

**Also save to GitHub (highly recommended)**

The spec specifically asks for a GitHub repo as a deliverable anyway, so set it up early:
```
File → Save a copy in GitHub


In [ ]:
!pip uninstall chromadb -y
!pip install chromadb==0.5.23 -q

In [ ]:
!pip install chromadb==0.5.23 faiss-cpu sentence-transformers \
             gradio transformers accelerate bitsandbytes \
             rank-bm25 langchain langchain-text-splitters \
             wikipedia-api pandas scikit-learn nltk -q

Install dependencies

In [ ]:
!pip install transformers torch accelerate bitsandbytes \
             sentence-transformers \
             faiss-cpu chromadb \
             rank-bm25 \
             langchain langchain-community nltk \
             gradio \
             pypdf beautifulsoup4 requests \
             wikipedia-api \
             langchain-text-splitters \
             pandas numpy tqdm scikit-learn -q

In [ ]:
import chromadb
print("chromadb:", chromadb.__version__)

Check if everything is actually installed correctly

In [ ]:
import transformers
import torch
import sentence_transformers
import faiss
import chromadb
import gradio
import langchain
from rank_bm25 import BM25Okapi
import pandas as pd
import sklearn
import wikipediaapi
import json
import os
from datetime import datetime

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
print("faiss version: OK")
print("chromadb:", chromadb.__version__)
print("gradio:", gradio.__version__)
print("langchain:", langchain.__version__)
print("pandas:", pd.__version__)
print("GPU available:", torch.cuda.is_available())

Define documents list with metadata

In [ ]:
DOCUMENTS = [
    {
        "title": "Rodent",
        "url": "https://en.wikipedia.org/wiki/Rodent",
        "category": "overview",
        "topic": "rodents"
    },
    {
        "title": "List of rodents",
        "url": "https://en.wikipedia.org/wiki/List_of_rodents",
        "category": "overview",
        "topic": "rodents"
    },
    {
        "title": "Beaver",
        "url": "https://en.wikipedia.org/wiki/Beaver",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Nutria",
        "url": "https://en.wikipedia.org/wiki/Nutria",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Gopher",
        "url": "https://en.wikipedia.org/wiki/Gopher",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Chinchilla",
        "url": "https://en.wikipedia.org/wiki/Chinchilla",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Squirrel",
        "url": "https://en.wikipedia.org/wiki/Squirrel",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Capybara",
        "url": "https://en.wikipedia.org/wiki/Capybara",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Guinea pig",
        "url": "https://en.wikipedia.org/wiki/Guinea_pig",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Marmot",
        "url": "https://en.wikipedia.org/wiki/Marmot",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Pika",
        "url": "https://en.wikipedia.org/wiki/Pika",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Rabbit",
        "url": "https://en.wikipedia.org/wiki/Rabbit",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Rat",
        "url": "https://en.wikipedia.org/wiki/Rat",
        "category": "species",
        "topic": "rodents"
    },
]

print(f"Defined {len(DOCUMENTS)} documents")

Fetch and normalize documents

In [ ]:
def fetch_wikipedia_page(wiki, title):
    """Fetch a single Wikipedia page and return its text."""
    page = wiki.page(title)
    if not page.exists():
        print(f"  WARNING: Page '{title}' not found")
        return None
    return page.text

In [ ]:
def normalize_text(text):
    """Basic text normalization."""
    # Remove excessive newlines
    lines = text.split('\n')
    # Filter out very short lines (usually section headers or empty)
    lines = [line.strip() for line in lines if len(line.strip()) > 30]
    return ' '.join(lines)

In [ ]:
def ingest_documents(document_list, save_path):
    """Fetch all documents from Wikipedia and save with metadata."""

    # Initialize Wikipedia API
    wiki = wikipediaapi.Wikipedia(
        language='en',
        user_agent='RodentRAGProject/1.0'
    )

    ingested = []
    failed = []

    for doc in document_list:
        print(f"Fetching: {doc['title']}...")

        try:
            raw_text = fetch_wikipedia_page(wiki, doc['title'])

            if raw_text is None:
                failed.append(doc['title'])
                continue

            clean_text = normalize_text(raw_text)

            # Build the full document object with metadata
            document = {
                "id": doc['title'].lower().replace(' ', '_'),
                "title": doc['title'],
                "text": clean_text,
                "metadata": {
                    "source": doc['url'],
                    "category": doc['category'],
                    "topic": doc['topic'],
                    "date_ingested": datetime.now().strftime("%Y-%m-%d"),
                    "char_count": len(clean_text),
                    "word_count": len(clean_text.split())
                }
            }

            ingested.append(document)
            print(f"  OK — {document['metadata']['word_count']} words")

        except Exception as e:
            print(f"  FAILED: {e}")
            failed.append(doc['title'])

    # Save to JSON file on Drive
    os.makedirs(save_path, exist_ok=True)
    output_file = os.path.join(save_path, 'documents.json')

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(ingested, f, indent=2, ensure_ascii=False)

    print(f"\n{'='*40}")
    print(f"Successfully ingested: {len(ingested)} documents")
    print(f"Failed: {len(failed)} documents {failed if failed else ''}")
    print(f"Saved to: {output_file}")

    return ingested



In [ ]:
# Run ingestion - saves to Google Drive
DOCS_SAVE_PATH = "/content/drive/MyDrive/rag_project/documents"


In [ ]:
documents = ingest_documents(DOCUMENTS, DOCS_SAVE_PATH)

Preview what you fetched

In [ ]:
def preview_documents(docs):
    """Print a summary table of all ingested documents."""

    rows = []
    for doc in docs:
        rows.append({
            "ID": doc['id'],
            "Title": doc['title'],
            "Category": doc['metadata']['category'],
            "Words": doc['metadata']['word_count'],
            "Chars": doc['metadata']['char_count'],
            "Date Ingested": doc['metadata']['date_ingested']
        })

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print(f"\nTotal words across all documents: {df['Words'].sum():,}")
    print(f"Average words per document: {df['Words'].mean():.0f}")

In [ ]:


preview_documents(documents)

Load documents back from Drive (for future sessions)

In [ ]:
def load_documents(save_path):
    """Load previously ingested documents from Drive."""
    output_file = os.path.join(save_path, 'documents.json')

    if not os.path.exists(output_file):
        print("No saved documents found. Run ingestion first.")
        return []

    with open(output_file, 'r', encoding='utf-8') as f:
        docs = json.load(f)

    print(f"Loaded {len(docs)} documents from {output_file}")
    return docs

# In future sessions, instead of re-fetching from Wikipedia,
# just call this:
# documents = load_documents(DOCS_SAVE_PATH)

# Phase **2**
Three types of chunking



First - we want to load our files from drive

In [ ]:
# This is a copy of code from phase 1
DOCS_SAVE_PATH = "/content/drive/MyDrive/rag_project/documents"


In [ ]:
def load_documents(save_path):
    """Load previously ingested documents from Drive."""
    output_file = os.path.join(save_path, 'documents.json')

    if not os.path.exists(output_file):
        print("No saved documents found. Run ingestion first.")
        return []

    with open(output_file, 'r', encoding='utf-8') as f:
        docs = json.load(f)

    print(f"Loaded {len(docs)} documents from {output_file}")
    return docs


In [ ]:
documents = load_documents(DOCS_SAVE_PATH)

In [ ]:
!pip install langchain-text-splitters -q

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

Function for splitting text in **fixed size chunking**

> later we will discuss how size of chunk and overlap affects model




In [ ]:
def fixed_size_chunking(documents, chunk_size=500, chunk_overlap=50):
    """
    Split documents into fixed size chunks.

    Parameters:
    - documents: list of document dicts from documents.json
    - chunk_size: how many characters per chunk
    - chunk_overlap: how many characters to repeat between chunks
    """

    splitter = CharacterTextSplitter(
        separator=" ",        # split at spaces so we dont cut mid-word
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,  # measure size in characters
    )

    all_chunks = []

    for doc in documents:
        # Split the document text into chunks
        chunks = splitter.split_text(doc['text'])

        # For each chunk, keep track of where it came from
        for i, chunk_text in enumerate(chunks):
            chunk = {
                "text": chunk_text,
                "metadata": {
                    "chunk_id": f"{doc['id']}_fixed_{i}",
                    "doc_id": doc['id'],
                    "title": doc['title'],
                    "source": doc['metadata']['source'],
                    "category": doc['metadata']['category'],
                    "chunk_index": i,
                    "total_chunks": len(chunks),
                    "chunk_size": chunk_size,
                    "chunk_overlap": chunk_overlap,
                    "chunking_strategy": "fixed_size"
                }
            }
            all_chunks.append(chunk)

    return all_chunks

Run it and see results

In [ ]:
# Run fixed size chunking on all documents
fixed_chunks = fixed_size_chunking(documents, chunk_size=500, chunk_overlap=50)

# Basic stats
print(f"Total chunks created: {len(fixed_chunks)}")
print(f"Average chunk length: {sum(len(c['text']) for c in fixed_chunks) / len(fixed_chunks):.0f} chars")
print(f"Shortest chunk: {min(len(c['text']) for c in fixed_chunks)} chars")
print(f"Longest chunk: {max(len(c['text']) for c in fixed_chunks)} chars")

Inspect a real chunk to see what it looks like

In [ ]:
# Look at first 3 chunks from the Capybara document
capybara_chunks = [c for c in fixed_chunks if c['metadata']['doc_id'] == 'capybara']

print(f"Capybara was split into {len(capybara_chunks)} chunks\n")
print("=" * 60)

for i, chunk in enumerate(capybara_chunks[:3]):
    print(f"\nCHUNK {i+1}:")
    print(f"Characters: {len(chunk['text'])}")
    print(f"Chunk ID: {chunk['metadata']['chunk_id']}")
    print(f"Text preview:")
    print(f"{chunk['text'][:200]}...")
    print("=" * 60)

Compare different chunk sizes

In [ ]:
# See how chunk size affects number of chunks produced
print("Chunk size comparison:")
print(f"{'Chunk Size':>12} | {'Total Chunks':>12} | {'Avg Length':>10}")
print("-" * 42)

for size in [256, 512, 1000]:
    chunks = fixed_size_chunking(documents, chunk_size=size, chunk_overlap=50)
    avg_len = sum(len(c['text']) for c in chunks) / len(chunks)
    print(f"{size:>12} | {len(chunks):>12} | {avg_len:>10.0f}")


Save fixed chunks to Drive

In [ ]:
import json
import os

CHUNKS_PATH = "/content/drive/MyDrive/rag_project/chunks"
os.makedirs(CHUNKS_PATH, exist_ok=True)

for size in [256, 512, 1000]:
    chunks = fixed_size_chunking(documents, chunk_size=size, chunk_overlap=50)

    filename = f"{CHUNKS_PATH}/fixed_chunks_{size}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(chunks)} chunks of size {size} → fixed_chunks_{size}.json")

Function for splitting text wirh **recursive chunking**

later we will discuss how size of chunk and overlap affects model

Import

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def recursive_chunking(documents, chunk_size=500, chunk_overlap=50):
    """
    Split documents using recursive character splitting.
    Tries to split at natural boundaries in this order:
    paragraphs -> lines -> sentences -> words -> characters

    Parameters:
    - documents: list of document dicts from documents.json
    - chunk_size: target size of each chunk in characters
    - chunk_overlap: characters to repeat between chunks
    """

    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )

    all_chunks = []

    for doc in documents:
        chunks = splitter.split_text(doc['text'])

        for i, chunk_text in enumerate(chunks):
            chunk = {
                "text": chunk_text,
                "metadata": {
                    "chunk_id": f"{doc['id']}_recursive_{i}",
                    "doc_id": doc['id'],
                    "title": doc['title'],
                    "source": doc['metadata']['source'],
                    "category": doc['metadata']['category'],
                    "chunk_index": i,
                    "total_chunks": len(chunks),
                    "chunk_size": chunk_size,
                    "chunk_overlap": chunk_overlap,
                    "chunking_strategy": "recursive"
                }
            }
            all_chunks.append(chunk)

    return all_chunks

separators=["\n\n", "\n", ". ", " ", ""] - explanation


It tries each separator in order

```
"\n\n" → first try to split at paragraph breaks
"\n"   → if still too big, split at line breaks  
". "   → if still too big, split at sentences
" "    → if still too big, split at words
""     → last resort, split at any character
```

Run and compare with fixed size

In [ ]:
recursive_chunks = recursive_chunking(documents, chunk_size=500, chunk_overlap=50)

print("=== RECURSIVE CHUNKING RESULTS ===")
print(f"Total chunks: {len(recursive_chunks)}")
print(f"Average length: {sum(len(c['text']) for c in recursive_chunks) / len(recursive_chunks):.0f} chars")
print(f"Shortest chunk: {min(len(c['text']) for c in recursive_chunks)} chars")
print(f"Longest chunk: {max(len(c['text']) for c in recursive_chunks)} chars")

print("\n=== COMPARISON WITH FIXED SIZE ===")
print(f"{'Method':>15} | {'Total Chunks':>12} | {'Avg Length':>10}")
print("-" * 45)
print(f"{'Fixed Size':>15} | {len(fixed_chunks):>12} | {sum(len(c['text']) for c in fixed_chunks)/len(fixed_chunks):>10.0f}")
print(f"{'Recursive':>15} | {len(recursive_chunks):>12} | {sum(len(c['text']) for c in recursive_chunks)/len(recursive_chunks):>10.0f}")

Compare a real chunk side by side

In [ ]:
# Look at same section of Capybara document in both methods
fixed_capybara = [c for c in fixed_chunks if c['metadata']['doc_id'] == 'capybara']
recursive_capybara = [c for c in recursive_chunks if c['metadata']['doc_id'] == 'capybara']

print("FIXED SIZE - Chunk 1:")
print(fixed_capybara[0]['text'])
print("\n" + "="*60)
print("\nRECURSIVE - Chunk 1:")
print(recursive_capybara[0]['text'])

Save three size of recursive chunking on drive

In [ ]:
for size in [256, 512, 1000]:
    chunks = recursive_chunking(documents, chunk_size=size, chunk_overlap=50)

    filename = f"{CHUNKS_PATH}/recursive_chunks_{size}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(chunks)} chunks of size {size} → recursive_chunks_{size}.json")


Function for splitting text wirh **semantical chunking**

later we will discuss how size of chunk and overlap affects model

 Install and import

In [ ]:
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer
import numpy as np
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

Load embedding model

 We WILL NOT use this model to embed chunks, we'll use it to embed sentances. After thah we'll know similarity between each sentance and we'll know how to divide them into chunks.

In [ ]:
print("Loading embedding model for semantic chunking...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

Semantic chunking function

In [ ]:
def semantic_chunking(documents, embedding_model, threshold=0.3, min_chunk_size=100, max_chunk_size=1000):
    """
    Split documents based on semantic similarity between sentences.
    When similarity drops significantly between consecutive sentences,
    that signals a topic change and we split there.

    Parameters:
    - documents: list of document dicts from documents.json
    - embedding_model: SentenceTransformer model for embeddings
    - threshold: similarity drop that triggers a split (lower = more splits)
    - min_chunk_size: minimum characters a chunk must have
    - max_chunk_size: maximum characters a chunk can have
    """

    def get_similarity(emb1, emb2):
        """Calculate cosine similarity between two embeddings."""
        return float(np.dot(emb1, emb2) /
                    (np.linalg.norm(emb1) * np.linalg.norm(emb2)))

    def split_into_chunks(sentences, embeddings, threshold, min_size, max_size):
        """Group sentences into chunks based on similarity drops."""

        if len(sentences) == 0:
            return []

        chunks = []
        current_chunk_sentences = [sentences[0]]
        current_chunk_size = len(sentences[0])

        for i in range(1, len(sentences)):
            current_sentence = sentences[i]
            sentence_size = len(current_sentence)

            # Calculate similarity between current and previous sentence
            similarity = get_similarity(embeddings[i-1], embeddings[i])

            # Decide whether to split or continue current chunk
            should_split = False

            # Split if similarity drops below threshold (topic change)
            if similarity < threshold:
                should_split = True

            # Force split if chunk is getting too large
            if current_chunk_size + sentence_size > max_size:
                should_split = True

            # Never split if chunk is too small
            if current_chunk_size < min_size:
                should_split = False

            if should_split:
                # Save current chunk and start new one
                chunks.append(' '.join(current_chunk_sentences))
                current_chunk_sentences = [current_sentence]
                current_chunk_size = sentence_size
            else:
                # Add sentence to current chunk
                current_chunk_sentences.append(current_sentence)
                current_chunk_size += sentence_size

        # Don't forget the last chunk
        if current_chunk_sentences:
            chunks.append(' '.join(current_chunk_sentences))

        return chunks

    all_chunks = []

    for doc_index, doc in enumerate(documents):
        print(f"Processing {doc['title']} ({doc_index + 1}/{len(documents)})...")

        # Step 1 - Split document into sentences
        sentences = sent_tokenize(doc['text'])

        # Remove very short sentences (noise)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 20]

        if len(sentences) == 0:
            print(f"  WARNING: No sentences found in {doc['title']}")
            continue

        # Step 2 - Embed all sentences at once (faster than one by one)
        embeddings = embedding_model.encode(sentences, show_progress_bar=False)

        # Step 3 - Group sentences into chunks based on similarity
        chunk_texts = split_into_chunks(
            sentences,
            embeddings,
            threshold=threshold,
            min_size=min_chunk_size,
            max_size=max_chunk_size
        )

        # Step 4 - Build chunk objects with metadata
        for i, chunk_text in enumerate(chunk_texts):
            chunk = {
                "text": chunk_text,
                "metadata": {
                    "chunk_id": f"{doc['id']}_semantic_{i}",
                    "doc_id": doc['id'],
                    "title": doc['title'],
                    "source": doc['metadata']['source'],
                    "category": doc['metadata']['category'],
                    "chunk_index": i,
                    "total_chunks": len(chunk_texts),
                    "threshold": threshold,
                    "chunking_strategy": "semantic"
                }
            }
            all_chunks.append(chunk)

        print(f"  Created {len(chunk_texts)} chunks")

    return all_chunks

Run semantic chunking

In [ ]:
print("Starting semantic chunking (this will take a few minutes)...")
print("It is slower than fixed/recursive because it embeds every sentence\n")

semantic_chunks = semantic_chunking(
    documents,
    embedding_model,
    threshold=0.3,
    min_chunk_size=100,
    max_chunk_size=1000
)

print(f"\n=== SEMANTIC CHUNKING RESULTS ===")
print(f"Total chunks: {len(semantic_chunks)}")
print(f"Average length: {sum(len(c['text']) for c in semantic_chunks) / len(semantic_chunks):.0f} chars")
print(f"Shortest chunk: {min(len(c['text']) for c in semantic_chunks)} chars")
print(f"Longest chunk: {max(len(c['text']) for c in semantic_chunks)} chars")

 Compare all three methods

In [ ]:
print("=== COMPARISON OF ALL THREE CHUNKING METHODS ===\n")
print(f"{'Method':>15} | {'Total Chunks':>12} | {'Avg Length':>10} | {'Min':>6} | {'Max':>6}")
print("-" * 60)

methods = [
    ("Fixed Size", fixed_chunks),
    ("Recursive", recursive_chunks),
    ("Semantic", semantic_chunks)
]

for name, chunks in methods:
    lengths = [len(c['text']) for c in chunks]
    print(f"{name:>15} | {len(chunks):>12} | {sum(lengths)/len(lengths):>10.0f} | {min(lengths):>6} | {max(lengths):>6}")


Look at real chunk

In [ ]:
semantic_capybara = [c for c in semantic_chunks if c['metadata']['doc_id'] == 'capybara']

print(f"Capybara was split into {len(semantic_capybara)} semantic chunks\n")

for i, chunk in enumerate(semantic_capybara[:3]):
    print(f"CHUNK {i+1} ({len(chunk['text'])} chars):")
    print(chunk['text'][:300])
    print("=" * 60)

Test different thresholds

In [ ]:
# Threshold controls how sensitive splitting is to topic changes
# Lower threshold = splits more often = more smaller chunks
# Higher threshold = splits less often = fewer larger chunks

print("Threshold comparison:")
print(f"{'Threshold':>12} | {'Total Chunks':>12} | {'Avg Length':>10}")
print("-" * 42)

for threshold in [0.2, 0.3, 0.5]:
    chunks = semantic_chunking(
        documents,
        embedding_model,
        threshold=threshold,
        min_chunk_size=100,
        max_chunk_size=1000
    )
    avg_len = sum(len(c['text']) for c in chunks) / len(chunks)
    print(f"{threshold:>12} | {len(chunks):>12} | {avg_len:>10.0f}")

Save to drive

In [ ]:
for threshold in [0.2, 0.3, 0.5]:
    chunks = semantic_chunking(
        documents,
        embedding_model,
        threshold=threshold,
        min_chunk_size=100,
        max_chunk_size=1000
    )

    filename = f"{CHUNKS_PATH}/semantic_chunks_{threshold}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(chunks)} chunks with threshold {threshold} → semantic_chunks_{threshold}.json")


# Phase 3
Embedding - we will do example on only 20 samples, so we can test which model is better. In the next phase when we implement vector database, we'll do this proces for all chunks.

Imports

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time
import json
import os
import psutil
import torch
import random

Load your chunks

In [ ]:
CHUNKS_PATH = "/content/drive/MyDrive/rag_project/chunks"

In [ ]:
with open(f"{CHUNKS_PATH}/recursive_chunks_512.json", 'r') as f:
    test_chunks = json.load(f)

In [ ]:
chunk_texts = [chunk['text'] for chunk in test_chunks]

# Sample 20 chunks for visualization (full matrix would be too large to read)
random.seed(42)
sample_indices = random.sample(range(len(chunk_texts)), 20)
sample_texts = [chunk_texts[i] for i in sample_indices]
sample_titles = [test_chunks[i]['metadata']['title'] for i in sample_indices]

print(f"Selected 20 chunks from {len(chunk_texts)} total chunks")
print("\nSelected chunks from documents:")
for i, title in enumerate(sample_titles):
    print(f"  Chunk {i:2d}: {title} - {sample_texts[i][:60]}...")

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading MiniLM...")
minilm = SentenceTransformer('all-MiniLM-L6-v2')
print("Loading BGE...")
bge = SentenceTransformer('BAAI/bge-small-en-v1.5')
print("Both models loaded!")

Compute similarity matrices for both models

In [ ]:
def compute_similarity_matrix(model, texts, model_name):
    """Embed all texts and compute full similarity matrix."""

    print(f"Computing similarity matrix for {model_name}...")

    # Embed all chunks
    embeddings = model.encode(texts, show_progress_bar=False, batch_size=32)

    # Normalize embeddings
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normalized = embeddings / norms

    # Compute full similarity matrix
    similarity_matrix = np.dot(normalized, normalized.T)

    print(f"  Matrix shape: {similarity_matrix.shape}")
    print(f"  Average similarity: {np.mean(similarity_matrix):.3f}")
    print(f"  Max off-diagonal similarity: {np.max(similarity_matrix - np.eye(len(texts))):.3f}")
    print(f"  Min similarity: {np.min(similarity_matrix):.3f}")

    return similarity_matrix, embeddings

In [ ]:
minilm_matrix, minilm_embs = compute_similarity_matrix(minilm, sample_texts, "all-MiniLM-L6-v2")
print()
bge_matrix, bge_embs = compute_similarity_matrix(bge, sample_texts, "BAAI/bge-small-en-v1.5")

Visualize similarity matrices

In [ ]:
import matplotlib.pyplot as plt
import os

# Create folder if it doesnt exist
RESULTS_PATH = "/content/drive/MyDrive/rag_project/eval_results"
os.makedirs(RESULTS_PATH, exist_ok=True)

# Create short labels for each chunk
labels = [f"{i}:{title[:10]}" for i, title in enumerate(sample_titles)]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, matrix, model_name in zip(
    axes,
    [minilm_matrix, bge_matrix],
    ["all-MiniLM-L6-v2", "BAAI/bge-small-en-v1.5"]
):
    im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_title(f"Similarity Matrix\n{model_name}", fontsize=12)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    plt.colorbar(im, ax=ax, label='Cosine Similarity')

plt.tight_layout()
plt.savefig(f"{RESULTS_PATH}/similarity_matrices.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved similarity matrix plot to Drive")

Show similarity for a specific query against all chunks

In [ ]:
def query_similarity_report(model, texts, titles, query, model_name):
    """Show how similar a query is to every chunk."""

    # Embed chunks and query
    chunk_embeddings = model.encode(texts, show_progress_bar=False)
    query_embedding = model.encode([query], show_progress_bar=False)[0]

    # Compute similarities
    norms = np.linalg.norm(chunk_embeddings, axis=1)
    query_norm = np.linalg.norm(query_embedding)
    similarities = np.dot(chunk_embeddings, query_embedding) / (norms * query_norm)

    # Sort by similarity
    sorted_indices = np.argsort(similarities)[::-1]

    print(f"\nModel: {model_name}")
    print(f"Query: '{query}'")
    print(f"\n{'Rank':>4} | {'Score':>6} | {'Document':>15} | Chunk Preview")
    print("-" * 80)

    for rank, idx in enumerate(sorted_indices[:10]):
        print(f"{rank+1:>4} | {similarities[idx]:>6.3f} | {titles[idx]:>15} | {texts[idx]}...")



In [ ]:
# Test with a real query
query = "What is the largest rodent in the world"

query_similarity_report(minilm, sample_texts, sample_titles, query, "all-MiniLM-L6-v2")
query_similarity_report(bge, sample_texts, sample_titles, query, "BAAI/bge-small-en-v1.5")

Compare how the two models rank chunks differently

In [ ]:
def compare_model_rankings(texts, titles, queries, model1, model1_name, model2, model2_name):
    """Show side by side how two models rank chunks for the same query."""

    for query in queries:
        print(f"\nQUERY: '{query}'")
        print(f"{'Rank':>4} | {'Score1':>7} | {'Model1 Top chunk':>30} | {'Score2':>7} | {'Model2 Top chunk'}")
        print("-" * 90)

        # Get rankings from both models
        embs1 = model1.encode(texts, show_progress_bar=False)
        embs2 = model2.encode(texts, show_progress_bar=False)
        q_emb1 = model1.encode([query], show_progress_bar=False)[0]
        q_emb2 = model2.encode([query], show_progress_bar=False)[0]

        sims1 = np.dot(embs1, q_emb1) / (np.linalg.norm(embs1, axis=1) * np.linalg.norm(q_emb1))
        sims2 = np.dot(embs2, q_emb2) / (np.linalg.norm(embs2, axis=1) * np.linalg.norm(q_emb2))

        top5_1 = np.argsort(sims1)[::-1][:5]
        top5_2 = np.argsort(sims2)[::-1][:5]

        for rank in range(5):
            idx1 = top5_1[rank]
            idx2 = top5_2[rank]
            chunk1_preview = f"{titles[idx1]}: {texts[idx1][:20]}"
            chunk2_preview = f"{titles[idx2]}: {texts[idx2][:20]}"
            print(f"{rank+1:>4} | {sims1[idx1]:>7.3f} | {chunk1_preview:>30} | {sims2[idx2]:>7.3f} | {chunk2_preview}")

test_queries = [
    "What is the largest rodent in the world",
    "How do beavers build dams",
    "Where do chinchillas come from",
]

compare_model_rankings(
    sample_texts, sample_titles,
    test_queries,
    minilm, "MiniLM",
    bge, "BGE"
)


The one advantage MiniLM has is speed — it is slightly faster to encode since it has fewer parameters. But for a RAG system retrieval quality matters far more than a few milliseconds difference in encoding time.
Decision: We will use BAAI/bge-small-en-v1.5 as our embedding model for the main pipeline.

# Phase 4
Loading onto vector database
FAIIS and ChromaDB

Imports

In [ ]:
import faiss
import chromadb
import numpy as np
import json
import os
import time
from sentence_transformers import SentenceTransformer

print("All imports successful!")

Load chunks and embedding model

In [ ]:
CHUNKS_PATH = "/content/drive/MyDrive/rag_project/chunks"
VECTOR_STORE_PATH = "/content/drive/MyDrive/rag_project/vector_stores"
RESULTS_PATH = "/content/drive/MyDrive/rag_project/eval_results"

with open(f"{CHUNKS_PATH}/recursive_chunks_512.json", 'r') as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

print("Loading BGE embedding model...")
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
print("Model loaded!")

Embed all chunks

In [ ]:
def embed_chunks(chunks, model):
    """Embed all chunks and return vectors."""

    print(f"Embedding {len(chunks)} chunks...")
    texts = [chunk['text'] for chunk in chunks]

    start = time.time()
    embeddings = model.encode(
        texts,
        show_progress_bar=True,
        batch_size=32,
        normalize_embeddings=True  # normalize for cosine similarity
    )
    elapsed = time.time() - start

    print(f"Embedding complete in {elapsed:.1f} seconds")
    print(f"Embedding shape: {embeddings.shape}")

    return embeddings

In [ ]:
chunk_embeddings = embed_chunks(chunks, embedding_model)

 Build FAISS index

In [ ]:
def build_faiss_index(embeddings, chunks):
    """
    Build FAISS index from embeddings.
    FAISS only stores vectors, not metadata.
    We keep a separate list to map index positions to chunk metadata.
    """

    dimension = embeddings.shape[1]  # 384 for BGE-small

    # Create FAISS index
    # IndexFlatIP = Inner Product (cosine similarity with normalized vectors)
    index = faiss.IndexFlatIP(dimension)

    # Add all embeddings to index
    index.add(embeddings.astype(np.float32))

    print(f"FAISS index built successfully")
    print(f"  Vectors stored: {index.ntotal}")
    print(f"  Vector dimension: {dimension}")

    return index

In [ ]:
faiss_index = build_faiss_index(chunk_embeddings, chunks)

FAISS search with metadata filtering

In [ ]:
def faiss_search(query, index, chunks, embedding_model, top_k=5,
                 filter_category=None, filter_doc_id=None):
    """
    Search FAISS index with optional metadata filtering.

    Since FAISS does not support metadata filtering natively,
    we retrieve more results than needed and filter afterwards.

    Parameters:
    - query: user question
    - index: FAISS index
    - chunks: list of chunk dicts (for metadata)
    - top_k: number of results to return
    - filter_category: only return chunks from this category
    - filter_doc_id: only return chunks from this document
    """

    # Embed the query
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype(np.float32)

    # Search more than top_k if filtering is applied
    search_k = top_k * 10 if (filter_category or filter_doc_id) else top_k

    # Search FAISS index
    scores, indices = index.search(query_embedding, search_k)

    # Build results with metadata
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:  # FAISS returns -1 for empty slots
            continue

        chunk = chunks[idx]

        # Apply metadata filters
        if filter_category and chunk['metadata']['category'] != filter_category:
            continue
        if filter_doc_id and chunk['metadata']['doc_id'] != filter_doc_id:
            continue

        results.append({
            "text": chunk['text'],
            "score": float(score),
            "metadata": chunk['metadata']
        })

        if len(results) >= top_k:
            break

    return results

In [ ]:
results = faiss_search(
    "What is the largest living rodent in the world",
    faiss_index, chunks, embedding_model, top_k=3
)

for i, r in enumerate(results):
    print(f"Result {i+1} (score: {r['score']:.3f})")
    print(f"  Document: {r['metadata']['title']}")
    print(f"  Category: {r['metadata']['category']}")
    print(f"  Text: {r['text']}...")
    print()

Test FAISS metadata filterin

In [ ]:
print("=== FAISS METADATA FILTERING TEST ===\n")

# Filter by category - only species documents
print("Filter by category='species':")
results_filtered = faiss_search(
    "What is the largest rodent in the world",
    faiss_index, chunks, embedding_model,
    top_k=3,
    filter_category="species"
)
for r in results_filtered:
    print(f"  {r['metadata']['title']} (category: {r['metadata']['category']}) - score: {r['score']:.3f}")

print()

# Filter by specific document
print("Filter by doc_id='capybara':")
results_doc = faiss_search(
    "How does this animal swim",
    faiss_index, chunks, embedding_model,
    top_k=3,
    filter_doc_id="capybara"
)
for r in results_doc:
    print(f"  {r['metadata']['title']} - score: {r['score']:.3f}")
    print(f"  {r['text'][:100]}...")

 Save FAISS index to Drive

In [ ]:
VECTOR_STORE_PATH = "/content/drive/MyDrive/rag_project/vector_stores"
os.makedirs(VECTOR_STORE_PATH, exist_ok=True)

# Save FAISS index
faiss.write_index(faiss_index, f"{VECTOR_STORE_PATH}/faiss_index.bin")

# Save chunks alongside index (needed for metadata)
with open(f"{VECTOR_STORE_PATH}/faiss_chunks.json", 'w') as f:
    json.dump(chunks, f, indent=2)

print("FAISS index saved to Drive")
print(f"  Index file: faiss_index.bin")
print(f"  Chunks file: faiss_chunks.json")

In [ ]:
import os
FAISS_INDEX_PATH = f"{VECTOR_STORE_PATH}/faiss_index.bin"
FAISS_CHUNKS_PATH = f"{VECTOR_STORE_PATH}/faiss_chunks.json"
os.makedirs(VECTOR_STORE_PATH, exist_ok=True)

if os.path.exists(FAISS_INDEX_PATH):
    print("Loading existing FAISS index...")
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    with open(FAISS_CHUNKS_PATH, 'r') as f:
        chunks = json.load(f)
    print(f"Loaded! Vectors: {faiss_index.ntotal}")
else:
    print("Building new FAISS index...")
    chunk_embeddings = embed_chunks(chunks, embedding_model)
    faiss_index = build_faiss_index(chunk_embeddings, chunks)
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    with open(FAISS_CHUNKS_PATH, 'w') as f:
        json.dump(chunks, f)
    print("Saved to Drive!")

Build ChromaDB index

In [ ]:
CHROMA_PATH = "/content/drive/MyDrive/rag_project/vector_stores/chroma_db"
os.makedirs(CHROMA_PATH, exist_ok=True)

# Initialize ChromaDB with persistent storage
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Delete collection if it already exists (clean start)
try:
    chroma_client.delete_collection("rodents")
    print("Deleted existing collection")
except:
    pass

# Create fresh collection
collection = chroma_client.create_collection(
    name="rodents",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity
)

print(f"ChromaDB collection created")
print(f"  Stored at: {CHROMA_PATH}")

Add chunks to ChromaDB

In [ ]:
def add_chunks_to_chroma(collection, chunks, embeddings, batch_size=100):
    """
    Add all chunks to ChromaDB collection.
    ChromaDB stores vectors, text AND metadata together.
    """

    total = len(chunks)
    print(f"Adding {total} chunks to ChromaDB...")

    for i in range(0, total, batch_size):
        batch_chunks = chunks[i:i+batch_size]
        batch_embeddings = embeddings[i:i+batch_size]

        collection.add(
            ids=[c['metadata']['chunk_id'] for c in batch_chunks],
            embeddings=batch_embeddings.tolist(),
            documents=[c['text'] for c in batch_chunks],
            metadatas=[{
                "doc_id": c['metadata']['doc_id'],
                "title": c['metadata']['title'],
                "source": c['metadata']['source'],
                "category": c['metadata']['category'],
                "chunk_index": c['metadata']['chunk_index'],
            } for c in batch_chunks]
        )

        print(f"  Added batch {i//batch_size + 1}/{(total+batch_size-1)//batch_size}")

    print(f"\nChromaDB collection now has {collection.count()} chunks")




In [ ]:
add_chunks_to_chroma(collection, chunks, chunk_embeddings)

ChromaDB search with metadata filtering

In [ ]:
def chroma_search(query, collection, embedding_model, top_k=5,
                  filter_category=None, filter_doc_id=None):
    """
    Search ChromaDB with optional metadata filtering.
    ChromaDB supports native metadata filtering via where clause.

    Parameters:
    - query: user question
    - collection: ChromaDB collection
    - top_k: number of results to return
    - filter_category: only return chunks from this category
    - filter_doc_id: only return chunks from this document
    """

    # Embed query
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()[0]

    # Build where clause for metadata filtering
    where = None
    if filter_category and filter_doc_id:
        where = {
            "$and": [
                {"category": {"$eq": filter_category}},
                {"doc_id": {"$eq": filter_doc_id}}
            ]
        }
    elif filter_category:
        where = {"category": {"$eq": filter_category}}
    elif filter_doc_id:
        where = {"doc_id": {"$eq": filter_doc_id}}

    # Search ChromaDB
    query_params = {
        "query_embeddings": [query_embedding],
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"]
    }
    if where:
        query_params["where"] = where

    results = collection.query(**query_params)

    # Format results
    formatted = []
    for text, metadata, distance in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        formatted.append({
            "text": text,
            "score": 1 - distance,  # convert distance to similarity
            "metadata": metadata
        })

    return formatted

In [ ]:
print("Testing ChromaDB search...\n")
results = chroma_search(
    "What is the largest rodent in the world",
    collection, embedding_model, top_k=3
)

for i, r in enumerate(results):
    print(f"Result {i+1} (score: {r['score']:.3f})")
    print(f"  Document: {r['metadata']['title']}")
    print(f"  Category: {r['metadata']['category']}")
    print(f"  Text: {r['text'][:150]}...")
    print()

Test ChromaDB metadata filtering

In [ ]:
print("=== CHROMADB METADATA FILTERING TEST ===\n")

# Filter by category
print("Filter by category='species':")
results_filtered = chroma_search(
    "What is the largest rodent in the world",
    collection, embedding_model,
    top_k=3,
    filter_category="species"
)
for r in results_filtered:
    print(f"  {r['metadata']['title']} (category: {r['metadata']['category']}) - score: {r['score']:.3f}")

print()

# Filter by specific document
print("Filter by doc_id='capybara':")
results_doc = chroma_search(
    "How does this animal swim",
    collection, embedding_model,
    top_k=3,
    filter_doc_id="capybara"
)
for r in results_doc:
    print(f"  {r['metadata']['title']} - score: {r['score']:.3f}")
    print(f"  {r['text'][:100]}...")

 Compare latency between FAISS and ChromaDB

In [ ]:
def benchmark_latency(queries, faiss_index, collection, chunks, embedding_model, runs=5):
    """Compare search speed between FAISS and ChromaDB."""

    faiss_times = []
    chroma_times = []

    for query in queries:
        # Benchmark FAISS
        times = []
        for _ in range(runs):
            start = time.time()
            faiss_search(query, faiss_index, chunks, embedding_model, top_k=5)
            times.append(time.time() - start)
        faiss_times.append(sum(times)/runs)

        # Benchmark ChromaDB
        times = []
        for _ in range(runs):
            start = time.time()
            chroma_search(query, collection, embedding_model, top_k=5)
            times.append(time.time() - start)
        chroma_times.append(sum(times)/runs)

    print("=== LATENCY COMPARISON ===\n")
    print(f"{'Query':>35} | {'FAISS (s)':>10} | {'ChromaDB (s)':>12}")
    print("-" * 65)

    for query, ft, ct in zip(queries, faiss_times, chroma_times):
        print(f"{query[:35]:>35} | {ft:>10.3f} | {ct:>12.3f}")

    print("-" * 65)
    print(f"{'Average':>35} | {sum(faiss_times)/len(faiss_times):>10.3f} | {sum(chroma_times)/len(chroma_times):>12.3f}")

    winner = "FAISS" if sum(faiss_times) < sum(chroma_times) else "ChromaDB"
    print(f"\nFaster vector store: {winner}")

test_queries = [
    "What is the largest rodent in the world",
    "How do beavers build their dams",
    "Where do chinchillas come from",
    "What do squirrels eat in winter",
    "Are rabbits considered rodents",
]

benchmark_latency(test_queries, faiss_index, collection, chunks, embedding_model)


In [ ]:
print("\nDemonstrating ChromaDB persistence...")
print("Disconnecting and reconnecting to ChromaDB...\n")

# Reconnect to existing ChromaDB
chroma_client2 = chromadb.PersistentClient(path=CHROMA_PATH)
collection2 = chroma_client2.get_collection("rodents")

print(f"Chunks still in collection: {collection2.count()}")

Save benchmark results

In [ ]:
import pandas as pd

vector_store_results = {
    "faiss": {
        "type": "in-memory",
        "vectors_stored": faiss_index.ntotal,
        "persistence": "manual - must save and reload index file",
        "metadata_filtering": "post-retrieval filtering",
    },
    "chromadb": {
        "type": "persistent",
        "vectors_stored": collection.count(),
        "persistence": "automatic - survives session restarts",
        "metadata_filtering": "native - built in where clause",
    }
}

with open(f"{RESULTS_PATH}/vector_store_benchmark.json", 'w') as f:
    json.dump(vector_store_results, f, indent=2)

print("Vector store benchmark results saved to Drive")
print("\nPhase 4 Complete!")
print("Your Drive now has:")
print("  vector_stores/faiss_index.bin")
print("  vector_stores/faiss_chunks.json")
print("  vector_stores/chroma_db/")

# Phase 5
LLM Selection & Loading

Install dependencies

In [ ]:
!pip install transformers accelerate bitsandbytes -q

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time
import json
import os

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Configure 4-bit quantization

In [ ]:
# 4-bit quantization configuration
# This reduces model size from ~3GB to ~1GB
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,              # load model in 4-bit precision
    bnb_4bit_compute_dtype=torch.float16,  # compute in float16
    bnb_4bit_use_double_quant=True, # extra compression
    bnb_4bit_quant_type="nf4"       # best quality 4-bit format
)

print("4-bit quantization config ready")
print("  Precision: 4-bit (nf4)")
print("  Compute dtype: float16")
print("  Double quantization: enabled")

 Load tokenizer and model

In [ ]:
MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

In [ ]:
# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",        # automatically use GPU if available
    trust_remote_code=True,
    torch_dtype=torch.float16
)


In [ ]:
model.eval()  # set to inference mode

print("\nModel loaded successfully!")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU memory free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1024**3:.2f} GB")

Basic generation function

In [ ]:
def generate_answer(prompt, model, tokenizer, max_new_tokens=512, temperature=0.1):
    """
    Generate answer from model given a prompt.

    Parameters:
    - prompt: full prompt including context and question
    - max_new_tokens: maximum length of generated answer
    - temperature: lower = more focused, higher = more creative
                   we use 0.1 for RAG to stay grounded
    """

    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    input_length = inputs['input_ids'].shape[1]

    # Generate response
    start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1  # avoid repeating itself
        )

    elapsed = time.time() - start

    # Decode only the new tokens (not the input)
    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip(), elapsed

print("Generation function ready")


Test basic generation

In [ ]:
# Simple test to verify model works
test_prompt = """You are a helpful assistant. Answer the question below.

Question: What is the capital of France?
Answer:"""

answer, elapsed = generate_answer(test_prompt, model, tokenizer)

print(f"Answer: {answer}")
print(f"Generation time: {elapsed:.2f} seconds")

RAG prompt template

In [ ]:
def build_rag_prompt(question, context_chunks):
    """
    Build a grounded RAG prompt for Qwen2.

    The prompt is engineered to:
    1. Keep the model grounded in context
    2. Force it to say when it doesnt know
    3. Prevent hallucination
    4. Ask for concise focused answers
    """

    # Format context chunks
    context_parts = []
    for i, chunk in enumerate(context_chunks):
        context_parts.append(
            f"[Source {i+1}: {chunk['metadata']['title']}]\n{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    prompt = f"""You are a helpful assistant that answers questions about rodents.
You must follow these rules strictly:
1. Answer ONLY using information from the provided context below
2. If the answer is not in the context, say exactly: "I cannot find this information in the available documents"
3. Always mention which source your answer comes from
4. Keep your answer concise and factual
5. Do not add information from your own knowledge

Context:
{context}

Question: {question}

Answer:"""

    return prompt

print("RAG prompt template ready")

Test RAG generation with real chunks

In [ ]:
# Test RAG generation with real chunks
test_question = "What is the largest rodent in the world?"

retrieved_chunks = faiss_search(
    test_question,
    faiss_index, chunks, embedding_model, top_k=3
)

print(f"Retrieved {len(retrieved_chunks)} chunks")
for i, chunk in enumerate(retrieved_chunks):
    print(f"  {i+1}. {chunk['metadata']['title']} (score: {chunk['score']:.3f})")

prompt = build_rag_prompt(test_question, retrieved_chunks)
answer, elapsed = generate_answer(prompt, model, tokenizer)

print(f"\nQuestion: {test_question}")
print(f"Answer: {answer}")
print(f"Generation time: {elapsed:.2f} seconds")

Test unanswerable question

In [ ]:
# Test unanswerable question
unanswerable_question = "What is the population of Tokyo?"

retrieved_chunks = faiss_search(
    unanswerable_question,
    faiss_index, chunks, embedding_model, top_k=3
)

prompt = build_rag_prompt(unanswerable_question, retrieved_chunks)
answer, elapsed = generate_answer(prompt, model, tokenizer)

print(f"Question: {unanswerable_question}")
print(f"Answer: {answer}")

# Phase 6
Baseline RAG Pipeline

Imports

In [ ]:
import torch
import json
import os
import time
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports successful!")

Load everything we built in previous phases

Paths

In [ ]:
CHUNKS_PATH = "/content/drive/MyDrive/rag_project/chunks"
VECTOR_STORE_PATH = "/content/drive/MyDrive/rag_project/vector_stores"
CHROMA_PATH = f"{VECTOR_STORE_PATH}/chroma_db"
RESULTS_PATH = "/content/drive/MyDrive/rag_project/eval_results"

Load embedding model

In [ ]:
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

Load LLM

In [ ]:
MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

In [ ]:
model.eval()
print(f"GPU memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Confidence indicator

In [ ]:
def compute_confidence(retrieved_chunks):
    """
    Compute confidence score based on retrieval similarity scores.

    Logic:
    - High confidence:   top score > 0.75  → model likely has good answer
    - Medium confidence: top score > 0.50  → model has partial information
    - Low confidence:    top score < 0.50  → model likely cannot answer well

    Returns:
    - confidence_label: "High", "Medium", or "Low"
    - confidence_score: float between 0 and 1
    - should_answer: boolean, False if we should refuse to answer
    """

    if not retrieved_chunks:
        return "Low", 0.0, False

    # Get top score from retrieved chunks
    top_score = retrieved_chunks[0]['score']
    avg_score = sum(c['score'] for c in retrieved_chunks) / len(retrieved_chunks)

    # Combined confidence using top score and average
    confidence_score = (top_score * 0.7) + (avg_score * 0.3)

    # Determine confidence label
    if confidence_score >= 0.75:
        confidence_label = "High"
        should_answer = True
    elif confidence_score >= 0.50:
        confidence_label = "Medium"
        should_answer = True
    else:
        confidence_label = "Low"
        should_answer = False

    return confidence_label, round(confidence_score, 3), should_answer

print("Confidence indicator ready")

Prompt builder

In [ ]:
def build_rag_prompt(question, context_chunks):
    """
    Build a carefully engineered prompt to keep Qwen2 grounded.

    Key prompt engineering decisions:
    1. Explicitly forbid using outside knowledge
    2. Give exact phrase to use when answer not found
    3. Number the sources so model can cite them
    4. Ask for concise factual answers
    5. Put question at the end after context
       (small models pay more attention to recent text)
    """

    # Format context with numbered sources
    context_parts = []
    for i, chunk in enumerate(context_chunks):
        context_parts.append(
            f"[Source {i+1} - {chunk['metadata']['title']}]\n{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    prompt = f"""You are a factual assistant that only answers questions about rodents using the provided sources.

STRICT RULES:
- Use ONLY the information provided in the sources below
- Do NOT use any outside knowledge
- If the answer is not in the sources, respond with exactly: "I cannot find this information in the available documents"
- Always cite which source number your answer comes from
- Keep answers concise and factual
- Do not repeat the question

SOURCES:
{context}

QUESTION: {question}

ANSWER:"""

    return prompt

Answer generator

In [ ]:
def generate_answer(prompt, model, tokenizer, max_new_tokens=256, temperature=0.1):
    """
    Generate answer from LLM given a prompt.

    We use low temperature (0.1) to keep answers
    focused and deterministic rather than creative.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    input_length = inputs['input_ids'].shape[1]

    start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    elapsed = time.time() - start

    # Decode only new tokens
    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip(), elapsed

Full RAG pipeline

In [ ]:
def rag_pipeline(
    question,
    top_k=5,
    filter_category=None,
    filter_doc_id=None,
    verbose=True
):
    """
    Full RAG pipeline using FAISS:
    query → embed → retrieve → confidence check → prompt → generate → return
    """

    total_start = time.time()

    if verbose:
        print(f"Question: {question}")
        print("=" * 60)

    # Step 1 - Retrieve relevant chunks using FAISS
    if verbose:
        print("Step 1: Retrieving relevant chunks...")

    retrieval_start = time.time()
    retrieved_chunks = faiss_search(
        question,
        faiss_index,
        chunks,
        embedding_model,
        top_k=top_k,
        filter_category=filter_category,
        filter_doc_id=filter_doc_id
    )
    retrieval_time = time.time() - retrieval_start

    if verbose:
        print(f"  Retrieved {len(retrieved_chunks)} chunks in {retrieval_time:.2f}s")
        for i, chunk in enumerate(retrieved_chunks):
            print(f"  {i+1}. {chunk['metadata']['title']} (score: {chunk['score']:.3f})")

    # Step 2 - Compute confidence
    if verbose:
        print("\nStep 2: Computing confidence...")

    confidence_label, confidence_score, should_answer = compute_confidence(retrieved_chunks)

    if verbose:
        print(f"  Confidence: {confidence_label} ({confidence_score:.3f})")
        print(f"  Should answer: {should_answer}")

    # Step 3 - Build prompt and generate answer
    if verbose:
        print("\nStep 3: Generating answer...")

    if not should_answer:
        answer = "I cannot find this information in the available documents"
        generation_time = 0
        if verbose:
            print("  Low confidence - skipping generation, returning refusal")
    else:
        prompt = build_rag_prompt(question, retrieved_chunks)
        answer, generation_time = generate_answer(prompt, model, tokenizer)

    total_time = time.time() - total_start

    # Build sources list for citation
    sources = []
    seen = set()
    for chunk in retrieved_chunks:
        title = chunk['metadata']['title']
        if title not in seen:
            sources.append({
                "title": title,
                "source": chunk['metadata']['source'],
                "score": chunk['score']
            })
            seen.add(title)

    result = {
        "question": question,
        "answer": answer,
        "confidence_label": confidence_label,
        "confidence_score": confidence_score,
        "sources": sources,
        "retrieval_time": round(retrieval_time, 3),
        "generation_time": round(generation_time, 3),
        "total_time": round(total_time, 3)
    }

    if verbose:
        print(f"\n{'='*60}")
        print(f"ANSWER: {answer}")
        print(f"\nConfidence: {confidence_label} ({confidence_score:.3f})")
        print(f"Sources used:")
        for s in sources:
            print(f"  - {s['title']} ({s['source']})")
        print(f"\nTiming:")
        print(f"  Retrieval: {retrieval_time:.3f}s")
        print(f"  Generation: {generation_time:.3f}s")
        print(f"  Total: {total_time:.3f}s")

    return result


Test with answerable questions, unanswerable question and metadata filtering

In [ ]:
print("TEST 1: Answerable question\n")
result1 = rag_pipeline("What is the largest rodent in the world?")

print("\nTEST 2: Species specific question\n")
result2 = rag_pipeline("Where do chinchillas originally come from?")

print("\nTEST 3: Behaviour question\n")
result3 = rag_pipeline("How do beavers build their dams?")

print("\nTEST 4: Unanswerable question\n")
result4 = rag_pipeline("What is the population of Tokyo?")

print("\nTEST 5: Partially related but unanswerable\n")
result5 = rag_pipeline("What do rodents eat on the moon?")

print("\nTEST 6: With doc filter\n")
result6 = rag_pipeline("Tell me about this animals habitat", filter_doc_id="beaver")

Save pipeline results

In [ ]:
all_results = [result1, result2, result3, result4, result5, result6]

os.makedirs(RESULTS_PATH, exist_ok=True)
with open(f"{RESULTS_PATH}/baseline_pipeline_tests.json", 'w') as f:
    json.dump(all_results, f, indent=2)

print("Pipeline test results saved to Drive")
print("\n=== BASELINE PIPELINE SUMMARY ===")
print(f"{'Question':>45} | {'Confidence':>10} | {'Total Time':>10}")
print("-" * 75)
for r in all_results:
    print(f"{r['question'][:45]:>45} | {r['confidence_label']:>10} | {r['total_time']:>9.2f}s")

# Phase 7
Hybrid search — BM25 + dense retrieval with score fusion



Cross-encoder re-ranker — cross-encoder/ms-marco-MiniLM-L-6-v2



Query transformation — rewriting/expansion using Qwen

Hybrid search — BM25 + dense retrieval with score

 Install and import

In [ ]:
!pip install rank-bm25 -q

from rank_bm25 import BM25Okapi
import numpy as np
import json
import time


 Load chunks for BM25

In [ ]:
with open(f"{CHUNKS_PATH}/recursive_chunks_512.json", 'r') as f:
    chunks = json.load(f)

chunk_texts = [chunk['text'] for chunk in chunks]

print(f"Loaded {len(chunks)} chunks")

Build BM25 index

In [ ]:
def build_bm25_index(chunks):
    """
    Build BM25 index from chunks.
    BM25 works on tokenized text (list of words).
    """

    print("Building BM25 index...")

    # Tokenize each chunk into words
    # lowercase everything so "Capybara" and "capybara" are the same
    tokenized_chunks = []
    for chunk in chunks:
        tokens = chunk['text'].lower().split()
        tokenized_chunks.append(tokens)

    # Build BM25 index
    bm25_index = BM25Okapi(tokenized_chunks)

    print(f"BM25 index built!")
    print(f"  Documents indexed: {len(tokenized_chunks)}")
    print(f"  Average tokens per chunk: {sum(len(t) for t in tokenized_chunks)/len(tokenized_chunks):.0f}")

    return bm25_index

In [ ]:
bm25_index = build_bm25_index(chunks)

BM25 search function

In [ ]:
def bm25_search(query, bm25_index, chunks, top_k=5):
    """
    Search using BM25 keyword matching.
    Returns top_k most relevant chunks with normalized scores.
    """

    # Tokenize query same way as chunks
    tokenized_query = query.lower().split()

    # Get BM25 scores for all chunks
    scores = bm25_index.get_scores(tokenized_query)

    # Normalize scores to 0-1 range
    max_score = max(scores) if max(scores) > 0 else 1
    normalized_scores = scores / max_score

    # Get top k indices
    top_indices = np.argsort(normalized_scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "text": chunks[idx]['text'],
            "score": round(float(normalized_scores[idx]), 3),
            "metadata": chunks[idx]['metadata']
        })

    return results


In [ ]:
bm25_results = bm25_search(
    "What is the largest rodent in the world",
    bm25_index, chunks, top_k=3
)

for i, r in enumerate(bm25_results):
    print(f"Result {i+1} (score: {r['score']:.3f})")
    print(f"  Document: {r['metadata']['title']}")
    print(f"  Text: {r['text'][:150]}...")
    print()

Dense search function (from ChromaDB)

In [ ]:
def dense_search(query, top_k=5, filter_category=None, filter_doc_id=None):
    """Dense search using FAISS."""
    return faiss_search(
        query, faiss_index, chunks, embedding_model,
        top_k=top_k,
        filter_category=filter_category,
        filter_doc_id=filter_doc_id
    )

Hybrid search with score fusion

In [ ]:
def hybrid_search(query, bm25_index, top_k=5, bm25_weight=0.4, dense_weight=0.6):
    """
    Combine BM25 and dense retrieval scores.

    We search more candidates than needed from both methods
    then fuse scores and return top_k final results.

    Parameters:
    - bm25_weight: how much to trust BM25 score (default 0.4)
    - dense_weight: how much to trust dense score (default 0.6)
    - weights sum to 1.0

    We give dense retrieval slightly more weight because
    it understands meaning better for our use case.
    """

    search_k = top_k * 3  # search more candidates for better fusion

    # Get results from both methods
    bm25_results = bm25_search(query, bm25_index, chunks, top_k=search_k)
    dense_results = dense_search(query, top_k=search_k)

    # Build score dictionaries keyed by chunk_id
    bm25_scores = {}
    for r in bm25_results:
        chunk_id = r['metadata']['chunk_id']
        bm25_scores[chunk_id] = r['score']

    dense_scores = {}
    for r in dense_results:
        chunk_id = r['metadata']['chunk_id']
        dense_scores[chunk_id] = r['score']

    # Collect all unique chunk ids from both results
    all_chunk_ids = set(bm25_scores.keys()) | set(dense_scores.keys())

    # Fuse scores
    # If a chunk only appears in one method, give 0 for the other
    fused = {}
    for chunk_id in all_chunk_ids:
        bm25_score = bm25_scores.get(chunk_id, 0.0)
        dense_score = dense_scores.get(chunk_id, 0.0)
        fused[chunk_id] = (bm25_weight * bm25_score) + (dense_weight * dense_score)

    # Sort by fused score
    sorted_ids = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]

    # Build final results
    # Create lookup dict for chunks by chunk_id
    chunks_by_id = {c['metadata']['chunk_id']: c for c in chunks}

    results = []
    for chunk_id, fused_score in sorted_ids:
        chunk = chunks_by_id.get(chunk_id)
        if chunk:
            results.append({
                "text": chunk['text'],
                "score": round(fused_score, 3),
                "bm25_score": round(bm25_scores.get(chunk_id, 0.0), 3),
                "dense_score": round(dense_scores.get(chunk_id, 0.0), 3),
                "metadata": chunk['metadata']
            })

    return results

Compare baseline vs hybrid

In [ ]:
def compare_baseline_vs_hybrid(query, top_k=5):
    """
    Side by side comparison of baseline dense retrieval
    vs hybrid search for the same query.
    """

    print(f"Query: '{query}'")
    print("=" * 70)

    # Baseline dense only
    dense_results = dense_search(query, top_k=top_k)

    # Hybrid
    hybrid_results = hybrid_search(query, bm25_index, top_k=top_k)

    print(f"\n{'Rank':>4} | {'BASELINE (Dense Only)':>35} | {'HYBRID (BM25 + Dense)':>35}")
    print("-" * 80)

    for i in range(top_k):
        baseline = dense_results[i] if i < len(dense_results) else None
        hybrid = hybrid_results[i] if i < len(hybrid_results) else None

        baseline_str = f"{baseline['metadata']['title']} ({baseline['score']:.3f})" if baseline else "N/A"
        hybrid_str = f"{hybrid['metadata']['title']} ({hybrid['score']:.3f})" if hybrid else "N/A"

        print(f"{i+1:>4} | {baseline_str:>35} | {hybrid_str:>35}")

    print()

# Test with different queries
compare_baseline_vs_hybrid("What is the largest rodent in the world")
compare_baseline_vs_hybrid("beaver dam construction")
compare_baseline_vs_hybrid("chinchilla fur trade history")

Show concrete example where hybrid helps

In [ ]:
print("=== EXAMPLE WHERE HYBRID HELPS ===\n")

query = "nutria coypu invasive species"

print("This query contains very specific keywords.")
print("Dense retrieval might miss exact term matches.")
print("BM25 will catch exact keyword matches.\n")

dense_only = dense_search(query, top_k=3)
hybrid = hybrid_search(query, bm25_index, top_k=3)

print("BASELINE (Dense only):")
for i, r in enumerate(dense_only):
    print(f"  {i+1}. {r['metadata']['title']} - score: {r['score']:.3f}")
    print(f"     {r['text'][:100]}...")

print("\nHYBRID (BM25 + Dense):")
for i, r in enumerate(hybrid):
    print(f"  {i+1}. {r['metadata']['title']} - score: {r['score']:.3f}")
    print(f"     BM25: {r['bm25_score']:.3f} | Dense: {r['dense_score']:.3f}")
    print(f"     {r['text'][:100]}...")

Benchmark latency

In [ ]:
def benchmark_hybrid_latency(queries, runs=3):
    """Compare speed of baseline vs hybrid search."""

    baseline_times = []
    hybrid_times = []

    for query in queries:
        # Baseline timing
        times = []
        for _ in range(runs):
            start = time.time()
            dense_search(query, top_k=5)
            times.append(time.time() - start)
        baseline_times.append(sum(times)/runs)

        # Hybrid timing
        times = []
        for _ in range(runs):
            start = time.time()
            hybrid_search(query, bm25_index, top_k=5)
            times.append(time.time() - start)
        hybrid_times.append(sum(times)/runs)

    print("=== LATENCY: BASELINE vs HYBRID ===\n")
    print(f"{'Query':>40} | {'Baseline':>10} | {'Hybrid':>10}")
    print("-" * 68)

    for query, bt, ht in zip(queries, baseline_times, hybrid_times):
        print(f"{query[:40]:>40} | {bt:>10.3f} | {ht:>10.3f}")

    print("-" * 68)
    print(f"{'Average':>40} | {sum(baseline_times)/len(baseline_times):>10.3f} | {sum(hybrid_times)/len(hybrid_times):>10.3f}")

test_queries = [
    "What is the largest rodent in the world",
    "How do beavers build their dams",
    "Where do chinchillas come from",
    "nutria invasive species",
    "squirrel food storage winter",
]

benchmark_hybrid_latency(test_queries)

**Cross-encoder re-ranker**

 Install and import

In [ ]:
!pip install sentence-transformers -q

from sentence_transformers import CrossEncoder
import time
import json
import numpy as np

print("Imports successful!")

 Load cross encoder model

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Reranking function

In [ ]:
def rerank_chunks(query, retrieved_chunks, cross_encoder, top_k=5):
    """
    Rerank retrieved chunks using cross-encoder.

    Steps:
    1. Take chunks already retrieved by dense/hybrid search
    2. Score each chunk paired with the query using cross-encoder
    3. Sort by new scores and return top_k

    Parameters:
    - query: original user question
    - retrieved_chunks: chunks from baseline or hybrid search
    - cross_encoder: loaded CrossEncoder model
    - top_k: how many reranked results to return
    """

    if not retrieved_chunks:
        return []

    # Build query-chunk pairs for cross-encoder
    # Cross-encoder reads BOTH query and chunk together
    pairs = [[query, chunk['text']] for chunk in retrieved_chunks]

    # Score all pairs
    start = time.time()
    scores = cross_encoder.predict(pairs)
    elapsed = time.time() - start

    # Attach cross-encoder scores to chunks
    for i, chunk in enumerate(retrieved_chunks):
        chunk['cross_encoder_score'] = round(float(scores[i]), 4)
        chunk['original_rank'] = i + 1

    # Sort by cross-encoder score
    reranked = sorted(
        retrieved_chunks,
        key=lambda x: x['cross_encoder_score'],
        reverse=True
    )[:top_k]

    # Add new rank
    for i, chunk in enumerate(reranked):
        chunk['reranked_rank'] = i + 1

    return reranked, elapsed

 Full pipeline with reranking

In [ ]:
def rag_pipeline_reranked(
    query,
    collection,
    bm25_index,
    chunks,
    embedding_model,
    cross_encoder,
    initial_k=20,
    final_k=5
):
    """
    RAG pipeline with reranking.

    Key idea:
    - Retrieve MORE candidates than needed (initial_k=20)
    - Rerank those 20 with cross-encoder
    - Keep only top final_k=5 after reranking

    Why retrieve 20 then rerank to 5?
    - Bi-encoder retrieves fast but imprecisely
    - Cross-encoder is slow but very precise
    - Running cross-encoder on all chunks would be too slow
    - Running it on just 20 candidates is fast enough
    """

    print(f"Query: {query}")
    print("=" * 60)

    # Step 1 - Retrieve initial candidates with hybrid search
    print(f"Step 1: Retrieving {initial_k} initial candidates...")
    start = time.time()

    initial_chunks = hybrid_search(
        query,
        bm25_index,
        collection,
        chunks,
        embedding_model,
        top_k=initial_k
    )
    retrieval_time = time.time() - start
    print(f"  Retrieved {len(initial_chunks)} candidates in {retrieval_time:.3f}s")

    # Step 2 - Rerank with cross-encoder
    print(f"Step 2: Reranking with cross-encoder...")
    reranked_chunks, rerank_time = rerank_chunks(
        query,
        initial_chunks,
        cross_encoder,
        top_k=final_k
    )
    print(f"  Reranked to {len(reranked_chunks)} chunks in {rerank_time:.3f}s")

    # Step 3 - Show ranking changes
    print(f"\nRanking changes after reranking:")
    print(f"{'Title':>20} | {'Original Rank':>13} | {'New Rank':>8} | {'CE Score':>8}")
    print("-" * 60)
    for chunk in reranked_chunks:
        moved = chunk['original_rank'] - chunk['reranked_rank']
        direction = f"↑{moved}" if moved > 0 else f"↓{abs(moved)}" if moved < 0 else "→"
        print(f"{chunk['metadata']['title']:>20} | {chunk['original_rank']:>13} | {chunk['reranked_rank']:>8} | {chunk['cross_encoder_score']:>8.4f} {direction}")

    return reranked_chunks, retrieval_time, rerank_time

Compare baseline vs reranked function

Test

In [ ]:
def compare_baseline_vs_reranked(query):
    """
    Side by side comparison of baseline retrieval
    vs retrieval with cross-encoder reranking.
    """

    print(f"Query: '{query}'")
    print("=" * 70)

    # Baseline - just dense search top 5
    baseline = dense_search(query, top_k=5)

    # Reranked - retrieve 20 then rerank to 5
    initial = hybrid_search(query, bm25_index, top_k=20)
    reranked, _ = rerank_chunks(query, initial, cross_encoder, top_k=5)

    print(f"\n{'Rank':>4} | {'BASELINE':>25} | {'RERANKED':>25} | {'CE Score':>8}")
    print("-" * 72)

    for i in range(5):
        b = baseline[i] if i < len(baseline) else None
        r = reranked[i] if i < len(reranked) else None

        b_str = f"{b['metadata']['title']} ({b['score']:.3f})" if b else "N/A"
        r_str = f"{r['metadata']['title']} ({r['score']:.3f})" if r else "N/A"
        ce_score = f"{r['cross_encoder_score']:.4f}" if r else "N/A"

        print(f"{i+1:>4} | {b_str:>25} | {r_str:>25} | {ce_score:>8}")
    print()

# Test calls
compare_baseline_vs_reranked("What is the largest rodent in the world")
compare_baseline_vs_reranked("How do beavers build their dams")
compare_baseline_vs_reranked("Where do chinchillas originally come from")


 Example where reranking helps

In [ ]:
print("=== EXAMPLE WHERE RERANKING HELPS ===\n")

query = "How much does a capybara weigh"

baseline = dense_search(query, top_k=5)
print("BASELINE top 5:")
for i, r in enumerate(baseline):
    print(f"  {i+1}. {r['metadata']['title']} - score: {r['score']:.3f}")
    print(f"     {r['text'][:120]}...")

print()

initial = hybrid_search(query, bm25_index, top_k=20)
reranked, _ = rerank_chunks(query, initial, cross_encoder, top_k=5)
print("RERANKED top 5:")
for i, r in enumerate(reranked):
    print(f"  {i+1}. {r['metadata']['title']} - CE score: {r['cross_encoder_score']:.4f}")
    print(f"     {r['text'][:120]}...")

Latency benchmark

In [ ]:
def benchmark_reranking_latency(queries, runs=3):
    """Compare latency of baseline vs hybrid vs reranked."""

    baseline_times = []
    hybrid_times = []
    reranked_times = []

    for query in queries:
        # Baseline
        times = []
        for _ in range(runs):
            start = time.time()
            dense_search(query, top_k=5)
            times.append(time.time() - start)
        baseline_times.append(sum(times)/runs)

        # Hybrid
        times = []
        for _ in range(runs):
            start = time.time()
            hybrid_search(query, bm25_index, top_k=5)
            times.append(time.time() - start)
        hybrid_times.append(sum(times)/runs)

        # Reranked
        times = []
        for _ in range(runs):
            start = time.time()
            initial = hybrid_search(query, bm25_index, top_k=20)
            rerank_chunks(query, initial, cross_encoder, top_k=5)
            times.append(time.time() - start)
        reranked_times.append(sum(times)/runs)

    print("=== LATENCY: BASELINE vs HYBRID vs RERANKED ===\n")
    print(f"{'Query':>40} | {'Baseline':>8} | {'Hybrid':>8} | {'Reranked':>8}")
    print("-" * 75)

    for query, bt, ht, rt in zip(queries, baseline_times, hybrid_times, reranked_times):
        print(f"{query[:40]:>40} | {bt:>8.3f} | {ht:>8.3f} | {rt:>8.3f}")

    print("-" * 75)
    avg_b = sum(baseline_times)/len(baseline_times)
    avg_h = sum(hybrid_times)/len(hybrid_times)
    avg_r = sum(reranked_times)/len(reranked_times)
    print(f"{'Average':>40} | {avg_b:>8.3f} | {avg_h:>8.3f} | {avg_r:>8.3f}")
    print(f"\nReranking overhead vs baseline: {(avg_r-avg_b):.3f}s extra")

test_queries = [
    "What is the largest rodent in the world",
    "How do beavers build their dams",
    "Where do chinchillas come from",
    "nutria invasive species",
    "squirrel food storage winter",
]

benchmark_reranking_latency(test_queries)

**Query transformation — rewriting/expansion using Qwen**

Imports

In [ ]:
import time
import json


Query rewriting function

In [ ]:
def rewrite_query(original_query, model, tokenizer):
    """
    Rewrite a vague or poorly worded query into a
    clearer more specific version using Qwen2.

    Best for:
    - Vague queries like "tell me about the big one"
    - Queries with pronouns like "where do they live"
    - Short ambiguous queries
    """

    prompt = f"""You are a search query optimizer. Your job is to rewrite the user's question into a clearer, more specific search query that will help find relevant information in a document database about rodents.

Rules:
- Make the query more specific and descriptive
- Remove ambiguous pronouns like "they", "it", "them"
- Keep the same meaning but use more precise language
- Return ONLY the rewritten query, nothing else
- Do not add any explanation or preamble

Original query: {original_query}
Rewritten query:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    rewritten = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Clean up the output
    rewritten = rewritten.strip().split('\n')[0].strip()

    return rewritten

Query expansion function

In [ ]:
def expand_query(original_query, model, tokenizer):
    """
    Expand a query by adding related terms and synonyms.
    This helps catch more relevant chunks that use
    different words to describe the same concept.

    Best for:
    - Short queries missing important context
    - Queries where synonyms might appear in documents
    """

    prompt = f"""You are a search query expander. Your job is to expand the user's search query by adding related terms, synonyms, and relevant keywords about rodents.

Rules:
- Add 3-5 related terms or synonyms
- Keep the original query words
- Separate all terms with spaces
- Return ONLY the expanded query as a single line
- Do not add explanation or punctuation

Original query: {original_query}
Expanded query:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    expanded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    expanded = expanded.strip().split('\n')[0].strip()

    return expanded


 Hypothetical document generation

In [ ]:
def generate_hypothetical_document(query, model, tokenizer):
    """
    Instead of searching with the question, generate what
    a perfect answer would look like and search with that.

    This is called HyDE - Hypothetical Document Embeddings.

    Why it works:
    - Questions and answers have different structure
    - Searching with an answer-like text finds answer-like chunks
    - Better semantic match between query and stored chunks
    """

    prompt = f"""Write a short factual paragraph that directly answers the following question about rodents. Write it as if it were from an encyclopedia article.

Question: {query}

Answer paragraph:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    hypothetical_doc = tokenizer.decode(new_tokens, skip_special_tokens=True)
    hypothetical_doc = hypothetical_doc.strip()

    return hypothetical_doc


Full query transformation pipeline

In [ ]:
def query_transformation_pipeline(
    original_query,
    model,
    tokenizer,
    method="rewrite",
    top_k=5
):
    """
    Full pipeline with query transformation.
    Parameters:
    - method: "rewrite", "expand", or "hyde"
    """

    print(f"Original query: '{original_query}'")
    print(f"Method: {method}")
    print("=" * 60)

    # Step 1 - Transform the query
    start = time.time()

    if method == "rewrite":
        transformed_query = rewrite_query(original_query, model, tokenizer)
        print(f"Rewritten query: '{transformed_query}'")
    elif method == "expand":
        transformed_query = expand_query(original_query, model, tokenizer)
        print(f"Expanded query: '{transformed_query}'")
    elif method == "hyde":
        transformed_query = generate_hypothetical_document(original_query, model, tokenizer)
        print(f"Hypothetical document:\n'{transformed_query[:200]}...'")

    transform_time = time.time() - start
    print(f"Transform time: {transform_time:.2f}s")

    # Step 2 - Retrieve using transformed query
    print("\nRetrieving with transformed query...")
    retrieval_start = time.time()

    retrieved = hybrid_search(transformed_query, bm25_index, top_k=top_k)
    retrieval_time = time.time() - retrieval_start

    print(f"Retrieved {len(retrieved)} chunks in {retrieval_time:.3f}s")

    return {
        "original_query": original_query,
        "transformed_query": transformed_query,
        "method": method,
        "results": retrieved,
        "transform_time": round(transform_time, 3),
        "retrieval_time": round(retrieval_time, 3)
    }

print("Query transformation pipeline ready!")

Compare all three methods

In [ ]:
def compare_all_transformations(query, model, tokenizer):
    """
    Compare baseline vs all three transformation methods
    for the same query.
    """

    print(f"\nQUERY: '{query}'")
    print("=" * 70)

    # Baseline
    baseline = hybrid_search(query, bm25_index, top_k=5)

    # Rewrite
    rewrite_result = query_transformation_pipeline(
        query, model, tokenizer, method="rewrite"
    )

    # Expand
    expand_result = query_transformation_pipeline(
        query, model, tokenizer, method="expand"
    )

    # HyDE
    hyde_result = query_transformation_pipeline(
        query, model, tokenizer, method="hyde"
    )

    # Print comparison
    print(f"\n{'Rank':>4} | {'Baseline':>20} | {'Rewrite':>20} | {'Expand':>20} | {'HyDE':>20}")
    print("-" * 95)

    for i in range(5):
        b = baseline[i]['metadata']['title'] if i < len(baseline) else "N/A"
        r = rewrite_result['results'][i]['metadata']['title'] if i < len(rewrite_result['results']) else "N/A"
        e = expand_result['results'][i]['metadata']['title'] if i < len(expand_result['results']) else "N/A"
        h = hyde_result['results'][i]['metadata']['title'] if i < len(hyde_result['results']) else "N/A"

        print(f"{i+1:>4} | {b:>20} | {r:>20} | {e:>20} | {h:>20}")

Test where rewriting helps

In [ ]:
print("=== EXAMPLE WHERE QUERY REWRITING HELPS ===\n")

vague_query = "where do they live and what do they eat"

print("This query has pronouns with no clear reference.")
print("Rewriting will add context about rodents.\n")

result = query_transformation_pipeline(
    vague_query, model, tokenizer, method="rewrite"
)

print("\nBaseline results:")
baseline = hybrid_search(vague_query, bm25_index, top_k=3)
for i, r in enumerate(baseline):
    print(f"  {i+1}. {r['metadata']['title']} - score: {r['score']:.3f}")

print("\nAfter rewriting results:")
for i, r in enumerate(result['results'][:3]):
    print(f"  {i+1}. {r['metadata']['title']} - score: {r['score']:.3f}")


 Test where expansion helps

In [ ]:
print("=== EXAMPLE WHERE QUERY EXPANSION HELPS ===\n")

short_query = "rodent teeth"

print("Short query might miss chunks using different terminology.")
print("Expansion adds related terms like incisors, gnawing, dental.\n")

result = query_transformation_pipeline(
    short_query, model, tokenizer, method="expand"
)

print("\nBaseline results:")
baseline = hybrid_search(short_query, bm25_index, top_k=3)
for i, r in enumerate(baseline):
    print(f"  {i+1}. {r['metadata']['title']} - score: {r['score']:.3f}")
    print(f"     {r['text'][:100]}...")

print("\nAfter expansion results:")
for i, r in enumerate(result['results'][:3]):
    print(f"  {i+1}. {r['metadata']['title']} - score: {r['score']:.3f}")
    print(f"     {r['text'][:100]}...")


Test HyDE

In [ ]:
print("=== EXAMPLE WHERE HYDE HELPS ===\n")

question = "What adaptations help capybaras survive in water"

print("HyDE generates a hypothetical answer paragraph")
print("and searches with that instead of the question.\n")

result = query_transformation_pipeline(
    question, model, tokenizer, method="hyde"
)

print("\nBaseline results:")
baseline = hybrid_search(question, bm25_index, top_k=3)
for i, r in enumerate(baseline):
    print(f"  {i+1}. {r['metadata']['title']} - score: {r['score']:.3f}")

print("\nAfter HyDE results:")
for i, r in enumerate(result['results'][:3]):
    print(f"  {i+1}. {r['metadata']['title']} - score: {r['score']:.3f}")

Run full comparison


In [ ]:
test_queries = [
    "where do they live",
    "rodent teeth",
    "What adaptations help capybaras survive in water",
]

for query in test_queries:
    compare_all_transformations(query, model, tokenizer)
    print("\n" + "="*70 + "\n")

# Phase 8
Evaluation Framework

Define 20 Q&A pairs

In [ ]:
# 15 answerable + 5 unanswerable questions
# For each answerable question we define:
# - question: the query
# - answer: expected answer
# - relevant_docs: which documents contain the answer
# - relevant_keywords: keywords that should appear in retrieved chunks

test_set = [
    # ============ ANSWERABLE QUESTIONS (15) ============
    {
        "id": "q1",
        "question": "What is the largest rodent in the world?",
        "expected_answer": "The capybara is the largest rodent in the world",
        "relevant_docs": ["capybara", "rodent"],
        "relevant_keywords": ["capybara", "largest"],
        "answerable": True
    },
    {
        "id": "q2",
        "question": "How much does a capybara weigh?",
        "expected_answer": "Capybaras can weigh up to 66 kg",
        "relevant_docs": ["capybara", "rodent"],
        "relevant_keywords": ["capybara", "kg", "weigh"],
        "answerable": True
    },
    {
        "id": "q3",
        "question": "Where do chinchillas originally come from?",
        "expected_answer": "Chinchillas are native to the Andes mountains in South America",
        "relevant_docs": ["chinchilla"],
        "relevant_keywords": ["chinchilla", "andes", "south america"],
        "answerable": True
    },
    {
        "id": "q4",
        "question": "What are beavers known for building?",
        "expected_answer": "Beavers are known for building dams and lodges",
        "relevant_docs": ["beaver"],
        "relevant_keywords": ["beaver", "dam", "lodge"],
        "answerable": True
    },
    {
        "id": "q5",
        "question": "Are rabbits considered rodents?",
        "expected_answer": "No rabbits are not rodents they are lagomorphs",
        "relevant_docs": ["rabbit"],
        "relevant_keywords": ["rabbit", "lagomorph", "rodent"],
        "answerable": True
    },
    {
        "id": "q6",
        "question": "What do squirrels do to prepare for winter?",
        "expected_answer": "Squirrels store food such as nuts and seeds for winter",
        "relevant_docs": ["squirrel"],
        "relevant_keywords": ["squirrel", "store", "food", "winter"],
        "answerable": True
    },
    {
        "id": "q7",
        "question": "What is a nutria?",
        "expected_answer": "Nutria is a large semi aquatic rodent native to South America",
        "relevant_docs": ["nutria"],
        "relevant_keywords": ["nutria", "rodent", "south america"],
        "answerable": True
    },
    {
        "id": "q8",
        "question": "How long do guinea pigs live?",
        "expected_answer": "Guinea pigs typically live 4 to 8 years",
        "relevant_docs": ["guinea_pig"],
        "relevant_keywords": ["guinea pig", "live", "years"],
        "answerable": True
    },
    {
        "id": "q9",
        "question": "What do marmots do in winter?",
        "expected_answer": "Marmots hibernate during winter months",
        "relevant_docs": ["marmot"],
        "relevant_keywords": ["marmot", "hibernate", "winter"],
        "answerable": True
    },
    {
        "id": "q10",
        "question": "What makes rodents different from other mammals?",
        "expected_answer": "Rodents have continuously growing incisor teeth that they must gnaw to keep short",
        "relevant_docs": ["rodent"],
        "relevant_keywords": ["incisor", "teeth", "gnaw", "growing"],
        "answerable": True
    },
    {
        "id": "q11",
        "question": "Where do pikas live?",
        "expected_answer": "Pikas live in mountainous regions and rocky areas",
        "relevant_docs": ["pika"],
        "relevant_keywords": ["pika", "mountain", "rock"],
        "answerable": True
    },
    {
        "id": "q12",
        "question": "What are rats known for in terms of intelligence?",
        "expected_answer": "Rats are highly intelligent and social animals capable of learning complex tasks",
        "relevant_docs": ["rat"],
        "relevant_keywords": ["rat", "intelligent", "social"],
        "answerable": True
    },
    {
        "id": "q13",
        "question": "What is special about chinchilla fur?",
        "expected_answer": "Chinchilla fur is extremely dense and soft making it valuable in the fur trade",
        "relevant_docs": ["chinchilla"],
        "relevant_keywords": ["chinchilla", "fur", "dense", "soft"],
        "answerable": True
    },
    {
        "id": "q14",
        "question": "How do capybaras behave socially?",
        "expected_answer": "Capybaras are highly social animals that live in groups",
        "relevant_docs": ["capybara"],
        "relevant_keywords": ["capybara", "social", "group"],
        "answerable": True
    },
    {
        "id": "q15",
        "question": "What is the second largest rodent in the world?",
        "expected_answer": "The beaver is the second largest rodent in the world",
        "relevant_docs": ["beaver", "rodent"],
        "relevant_keywords": ["beaver", "second", "largest"],
        "answerable": True
    },
    # ============ UNANSWERABLE QUESTIONS (5) ============
    {
        "id": "q16",
        "question": "What is the population of Brazil?",
        "expected_answer": "I cannot find this information in the available documents",
        "relevant_docs": [],
        "relevant_keywords": [],
        "answerable": False
    },
    {
        "id": "q17",
        "question": "Who invented the telephone?",
        "expected_answer": "I cannot find this information in the available documents",
        "relevant_docs": [],
        "relevant_keywords": [],
        "answerable": False
    },
    {
        "id": "q18",
        "question": "What is the speed of light?",
        "expected_answer": "I cannot find this information in the available documents",
        "relevant_docs": [],
        "relevant_keywords": [],
        "answerable": False
    },
    {
        "id": "q19",
        "question": "How do you make pasta carbonara?",
        "expected_answer": "I cannot find this information in the available documents",
        "relevant_docs": [],
        "relevant_keywords": [],
        "answerable": False
    },
    {
        "id": "q20",
        "question": "What is the capital of Japan?",
        "expected_answer": "I cannot find this information in the available documents",
        "relevant_docs": [],
        "relevant_keywords": [],
        "answerable": False
    },
]

print(f"Test set created!")
print(f"Total questions: {len(test_set)}")
print(f"Answerable: {sum(1 for q in test_set if q['answerable'])}")
print(f"Unanswerable: {sum(1 for q in test_set if not q['answerable'])}")

 Retrieval metrics functions

In [ ]:
def precision_at_k(retrieved_chunks, relevant_keywords, k):
    """
    Of the top k retrieved chunks how many are relevant?
    Relevance determined by keyword matching.
    """
    if not retrieved_chunks or not relevant_keywords:
        return 0.0

    top_k = retrieved_chunks[:k]
    relevant_count = 0

    for chunk in top_k:
        chunk_text = chunk['text'].lower()
        chunk_title = chunk['metadata']['title'].lower()
        combined = chunk_text + " " + chunk_title

        if any(kw.lower() in combined for kw in relevant_keywords):
            relevant_count += 1

    return round(relevant_count / k, 3)

def recall_at_k(retrieved_chunks, relevant_docs, k):
    """
    Of all relevant documents that exist,
    how many did we retrieve in top k?
    """
    if not retrieved_chunks or not relevant_docs:
        return 0.0

    top_k = retrieved_chunks[:k]
    retrieved_doc_ids = set(
        chunk['metadata']['doc_id'] for chunk in top_k
    )

    relevant_found = len(
        set(relevant_docs) & retrieved_doc_ids
    )

    return round(relevant_found / len(relevant_docs), 3)

def mean_reciprocal_rank(retrieved_chunks, relevant_docs):
    """
    Where is the first relevant chunk in the ranking?
    Higher is better.
    """
    if not retrieved_chunks or not relevant_docs:
        return 0.0

    for i, chunk in enumerate(retrieved_chunks):
        if chunk['metadata']['doc_id'] in relevant_docs:
            return round(1.0 / (i + 1), 3)

    return 0.0

LLM as judge function

In [ ]:
def llm_judge(question, generated_answer, expected_answer,
              model, tokenizer):
    """
    Use Qwen2 to judge the quality of a generated answer.

    Scores three dimensions:
    - Correctness: Is the answer factually correct?
    - Groundedness: Is the answer based on the context?
    - Refusal: Did the model correctly refuse unanswerable questions?

    Returns scores from 0 to 3 for each dimension.
    """

    prompt = f"""You are an evaluation judge. Score the following answer on three criteria.

Question: {question}
Expected Answer: {expected_answer}
Generated Answer: {generated_answer}

Score each criterion from 0 to 3:
- Correctness: Does the answer contain correct information? (0=wrong, 1=partially correct, 2=mostly correct, 3=fully correct)
- Groundedness: Is the answer based on facts without hallucination? (0=hallucinated, 1=mostly hallucinated, 2=mostly grounded, 3=fully grounded)
- Appropriate Refusal: If the question is unanswerable did the model correctly refuse? (0=should have refused but didnt, 1=unclear, 2=partially refused, 3=correctly refused or correctly answered)

Respond in this exact format only:
Correctness: [0-3]
Groundedness: [0-3]
Refusal: [0-3]"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Parse scores from response
    scores = {"correctness": 0, "groundedness": 0, "refusal": 0}

    try:
        for line in response.strip().split('\n'):
            if 'Correctness:' in line:
                scores['correctness'] = int(line.split(':')[1].strip()[0])
            elif 'Groundedness:' in line:
                scores['groundedness'] = int(line.split(':')[1].strip()[0])
            elif 'Refusal:' in line:
                scores['refusal'] = int(line.split(':')[1].strip()[0])
    except:
        # Fallback rule based scoring if parsing fails
        answer_lower = generated_answer.lower()
        expected_lower = expected_answer.lower()

        # Check correctness by keyword overlap
        expected_words = set(expected_lower.split())
        answer_words = set(answer_lower.split())
        overlap = len(expected_words & answer_words) / len(expected_words)
        scores['correctness'] = min(3, int(overlap * 3))

        # Check groundedness
        scores['groundedness'] = 2

        # Check refusal
        if "cannot find" in answer_lower or "not in" in answer_lower:
            scores['refusal'] = 3
        else:
            scores['refusal'] = 2

    return scores

 Single pipeline evaluator

In [ ]:
def evaluate_pipeline(
    test_set,
    pipeline_name,
    pipeline_fn,
    model,
    tokenizer,
    k_values=[1, 3, 5]
):
    """
    Evaluate a single pipeline variant on the full test set.

    Parameters:
    - pipeline_fn: function that takes a question and returns retrieved chunks
    - k_values: list of k values to evaluate at
    """

    print(f"\nEvaluating pipeline: {pipeline_name}")
    print("=" * 60)

    results = []

    for i, qa in enumerate(test_set):
        print(f"  Question {i+1}/{len(test_set)}: {qa['question'][:50]}...")

        # Get retrieved chunks from pipeline
        retrieved = pipeline_fn(qa['question'])

        # Compute retrieval metrics
        metrics = {}
        for k in k_values:
            metrics[f'precision@{k}'] = precision_at_k(
                retrieved, qa['relevant_keywords'], k
            )
            metrics[f'recall@{k}'] = recall_at_k(
                retrieved, qa['relevant_docs'], k
            )

        metrics['mrr'] = mean_reciprocal_rank(
            retrieved, qa['relevant_docs']
        )

        # Generate answer using RAG
        if retrieved:
            prompt = build_rag_prompt(qa['question'], retrieved[:3])
            generated_answer, _ = generate_answer(prompt, model, tokenizer)
        else:
            generated_answer = "I cannot find this information in the available documents"

        # Judge answer quality
        judge_scores = llm_judge(
            qa['question'],
            generated_answer,
            qa['expected_answer'],
            model, tokenizer
        )

        result = {
            "question_id": qa['id'],
            "question": qa['question'],
            "answerable": qa['answerable'],
            "generated_answer": generated_answer,
            "expected_answer": qa['expected_answer'],
            **metrics,
            **judge_scores
        }

        results.append(result)

        print(f"    P@3: {metrics['precision@3']:.2f} | R@3: {metrics['recall@3']:.2f} | MRR: {metrics['mrr']:.2f} | Correctness: {judge_scores['correctness']}/3")

    return results


 Define pipeline variants

In [ ]:
def baseline_pipeline_fn(question):
    """Baseline: dense search only"""
    return dense_search(question, top_k=5)

def hybrid_pipeline_fn(question):
    """Hybrid: BM25 + dense search"""
    return hybrid_search(question, bm25_index, top_k=5)

def reranked_pipeline_fn(question):
    """Hybrid + cross-encoder reranking"""
    initial = hybrid_search(question, bm25_index, top_k=20)
    reranked, _ = rerank_chunks(question, initial, cross_encoder, top_k=5)
    return reranked

def rewrite_pipeline_fn(question):
    """Hybrid + query rewriting"""
    rewritten = rewrite_query(question, model, tokenizer)
    return hybrid_search(rewritten, bm25_index, top_k=5)

def full_optimized_pipeline_fn(question):
    """Full optimized: rewrite + hybrid + rerank"""
    rewritten = rewrite_query(question, model, tokenizer)
    initial = hybrid_search(rewritten, bm25_index, top_k=20)
    reranked, _ = rerank_chunks(question, initial, cross_encoder, top_k=5)
    return reranked

pipeline_variants = {
    "Baseline": baseline_pipeline_fn,
    "Hybrid": hybrid_pipeline_fn,
    "Reranked": reranked_pipeline_fn,
    "QueryRewrite": rewrite_pipeline_fn,
    "FullOptimized": full_optimized_pipeline_fn,
}

print("Pipeline variants defined!")
for name in pipeline_variants:
    print(f"  - {name}")

Run full evaluation

this cell is added because I deleted ChromaDb content

In [ ]:
import faiss
import numpy as np

FAISS_INDEX_PATH = f"{VECTOR_STORE_PATH}/faiss_index.bin"
FAISS_CHUNKS_PATH = f"{VECTOR_STORE_PATH}/faiss_chunks.json"

if os.path.exists(FAISS_INDEX_PATH):
    print("Loading existing FAISS index...")
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    with open(FAISS_CHUNKS_PATH, 'r') as f:
        chunks = json.load(f)
    print(f"Loaded! Vectors: {faiss_index.ntotal}, Chunks: {len(chunks)}")
    print(f"Match: {faiss_index.ntotal == len(chunks)}")
else:
    print("FAISS index not found, rebuilding...")
    with open(f"{CHUNKS_PATH}/recursive_chunks_512.json", 'r') as f:
        chunks = json.load(f)
    texts = [chunk['text'] for chunk in chunks]
    embeddings = embedding_model.encode(
        texts, show_progress_bar=True,
        batch_size=32, normalize_embeddings=True
    ).astype(np.float32)
    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dimension)
    faiss_index.add(embeddings)
    os.makedirs(VECTOR_STORE_PATH, exist_ok=True)
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    with open(FAISS_CHUNKS_PATH, 'w') as f:
        json.dump(chunks, f)
    print(f"Built and saved! Vectors: {faiss_index.ntotal}")

# Build BM25 index
from rank_bm25 import BM25Okapi
tokenized_chunks = [chunk['text'].lower().split() for chunk in chunks]
bm25_index = BM25Okapi(tokenized_chunks)
print(f"BM25 index built with {len(chunks)} chunks")

In [ ]:
import os

all_eval_results = {}
RESULTS_PATH = "/content/drive/MyDrive/rag_project/eval_results"
os.makedirs(RESULTS_PATH, exist_ok=True)

for pipeline_name, pipeline_fn in pipeline_variants.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING: {pipeline_name}")
    print(f"{'='*60}")

    results = evaluate_pipeline(
        test_set,
        pipeline_name,
        pipeline_fn,
        model,
        tokenizer
    )

    all_eval_results[pipeline_name] = results

    # Save individual results
    with open(f"{RESULTS_PATH}/eval_{pipeline_name.lower()}.json", 'w') as f:
        json.dump(results, f, indent=2)

    print(f"\nSaved results for {pipeline_name}")

print("\nFull evaluation complete!")

Compute summary statistics

In [ ]:
import pandas as pd

def compute_summary(results):
    """Compute average metrics across all questions."""

    metrics = [
        'precision@1', 'precision@3', 'precision@5',
        'recall@1', 'recall@3', 'recall@5',
        'mrr', 'correctness', 'groundedness', 'refusal'
    ]

    summary = {}
    for metric in metrics:
        values = [r[metric] for r in results if metric in r]
        summary[metric] = round(sum(values) / len(values), 3) if values else 0

    return summary

# Compute summaries for all pipelines
summaries = {}
for pipeline_name, results in all_eval_results.items():
    summaries[pipeline_name] = compute_summary(results)

print("Summaries computed!")

 Print comparison table

In [ ]:
print("\n" + "="*80)
print("FULL EVALUATION COMPARISON TABLE")
print("="*80)

# Build dataframe
rows = []
for pipeline_name, summary in summaries.items():
    row = {"Pipeline": pipeline_name}
    row.update(summary)
    rows.append(row)

df = pd.DataFrame(rows)
df = df.set_index("Pipeline")

# Print retrieval metrics
print("\nRETRIEVAL METRICS:")
retrieval_cols = ['precision@1', 'precision@3', 'precision@5',
                  'recall@1', 'recall@3', 'recall@5', 'mrr']
print(df[retrieval_cols].to_string())

# Print answer quality metrics
print("\nANSWER QUALITY METRICS (0-3 scale):")
quality_cols = ['correctness', 'groundedness', 'refusal']
print(df[quality_cols].to_string())

# Save to Drive
df.to_csv(f"{RESULTS_PATH}/evaluation_comparison.csv")
print("\nComparison table saved to Drive!")

Print best pipeline recommendation

In [ ]:
print("\n" + "="*60)
print("PIPELINE RECOMMENDATION")
print("="*60)

# Find best pipeline for each metric
best_p3 = df['precision@3'].idxmax()
best_mrr = df['mrr'].idxmax()
best_correctness = df['correctness'].idxmax()
best_overall = df[['precision@3', 'recall@3', 'mrr',
                    'correctness', 'groundedness']].mean(axis=1).idxmax()

print(f"Best Precision@3:  {best_p3}")
print(f"Best MRR:          {best_mrr}")
print(f"Best Correctness:  {best_correctness}")
print(f"Best Overall:      {best_overall}")

# Save final summary
final_summary = {
    "evaluation_summary": summaries,
    "best_pipeline": {
        "precision@3": best_p3,
        "mrr": best_mrr,
        "correctness": best_correctness,
        "overall": best_overall
    }
}

with open(f"{RESULTS_PATH}/final_evaluation_summary.json", 'w') as f:
    json.dump(final_summary, f, indent=2)

print("\nFinal evaluation summary saved to Drive!")
print("\nPhase 8 Complete!")

# Phase 9
WEB UI

Instalation and imports

In [ ]:
!pip install gradio -q


In [ ]:
import gradio as gr
import json
import os
import faiss
import numpy as np
import wikipediaapi
from datetime import datetime

Build Gradio UI

In [ ]:
def build_ui():

    with gr.Blocks(
        title="Rodent RAG System",
        theme=gr.themes.Soft()
    ) as demo:

        gr.Markdown("""
        # 🐭 Rodent Knowledge Base — RAG System
        Ask questions about rodents and get answers grounded in Wikipedia sources.
        """)

        # ============ MAIN TAB ============
        with gr.Tab("🔍 Ask a Question"):

            with gr.Row():
                with gr.Column(scale=2):
                    question_input = gr.Textbox(
                        label="Your Question",
                        placeholder="e.g. What is the largest rodent in the world?",
                        lines=2
                    )

                    with gr.Row():
                        use_hybrid = gr.Checkbox(
                            label="Hybrid Search (BM25 + Dense)",
                            value=True
                        )
                        use_reranking = gr.Checkbox(
                            label="Cross-Encoder Reranking",
                            value=True
                        )
                        use_rewrite = gr.Checkbox(
                            label="Query Rewriting",
                            value=False
                        )

                    submit_btn = gr.Button(
                        "🔍 Search",
                        variant="primary",
                        size="lg"
                    )

                with gr.Column(scale=1):
                    confidence_output = gr.Textbox(
                        label="🎯 Confidence",
                        interactive=False
                    )
                    gr.Markdown("**Confidence levels:**")
                    gr.Markdown("🟢 High (>0.75) — Strong match found")
                    gr.Markdown("🟡 Medium (>0.50) — Partial match found")
                    gr.Markdown("🔴 Low (<0.50) — No good match found")

            answer_output = gr.Textbox(
                label="📝 Answer",
                lines=6,
                interactive=False
            )

            sources_output = gr.Markdown(
                label="📚 Sources"
            )

            gr.Markdown("### 💡 Example Questions")
            with gr.Row():
                gr.Examples(
                    examples=[
                        ["What is the largest rodent in the world?"],
                        ["How do beavers build their dams?"],
                        ["Where do chinchillas originally come from?"],
                        ["Are rabbits considered rodents?"],
                        ["What do marmots do in winter?"],
                        ["What is the population of Tokyo?"],
                    ],
                    inputs=question_input
                )

            submit_btn.click(
                fn=query_rag,
                inputs=[question_input, use_hybrid, use_reranking, use_rewrite],
                outputs=[answer_output, confidence_output, sources_output]
            )

            question_input.submit(
                fn=query_rag,
                inputs=[question_input, use_hybrid, use_reranking, use_rewrite],
                outputs=[answer_output, confidence_output, sources_output]
            )

        # ============ INFO TAB ============
        with gr.Tab("ℹ️ System Info"):
            gr.Markdown(f"""
            ## System Information

            **Model:** Qwen2-1.5B-Instruct (4-bit quantized)

            **Embedding Model:** BAAI/bge-small-en-v1.5

            **Vector Store:** FAISS (IndexFlatIP)

            **Chunking Strategy:** Recursive Character Splitting (512 chars, 50 overlap)

            **Advanced Techniques Available:**
            - ✅ Hybrid Search (BM25 + Dense)
            - ✅ Cross-Encoder Reranking (ms-marco-MiniLM-L-6-v2)
            - ✅ Query Rewriting (Qwen2)

            **Knowledge Base:**
            - Documents: 13 Wikipedia articles about rodents
            - Total chunks: {len(chunks)}

            **How to use:**
            1. Type your question in the search box
            2. Toggle search options as desired
            3. Click Search or press Enter
            4. View answer with confidence score and source citations
            """)

    return demo

In [ ]:
# Provjeri koje funkcije postoje
funcs = ['query_rag', 'dense_search', 'hybrid_search',
         'rerank_chunks', 'rewrite_query', 'compute_confidence',
         'build_rag_prompt', 'generate_answer']

for f in funcs:
    try:
        eval(f)
        print(f"OK: {f}")
    except NameError:
        print(f"MISSING: {f}")

Launch

In [ ]:
def query_rag(question, use_hybrid, use_reranking, use_query_rewrite):
    """Main function called by Gradio UI."""
    if not question.strip():
        return "Please enter a question.", "N/A", "No sources"

    try:
        # Step 1 - Transform query if enabled
        search_query = question
        if use_query_rewrite:
            search_query = rewrite_query(question, model, tokenizer)

        # Step 2 - Retrieve chunks
        if use_hybrid:
            retrieved = hybrid_search(search_query, bm25_index, top_k=20 if use_reranking else 5)
        else:
            retrieved = dense_search(search_query, top_k=20 if use_reranking else 5)

        # Step 3 - Rerank if enabled
        if use_reranking and retrieved:
            retrieved, _ = rerank_chunks(question, retrieved, cross_encoder, top_k=5)

        # Step 4 - Compute confidence
        confidence_label, confidence_score, should_answer = compute_confidence(retrieved)

        # Step 5 - Generate answer
        if not should_answer or not retrieved:
            answer = "I cannot find this information in the available documents."
        else:
            prompt = build_rag_prompt(question, retrieved[:3])
            answer, _ = generate_answer(prompt, model, tokenizer)

        # Step 6 - Format sources
        seen = set()
        source_lines = []
        for chunk in retrieved[:5]:
            title = chunk['metadata']['title']
            if title not in seen:
                source_lines.append(
                    f"**{title}** (relevance: {chunk['score']:.3f})\n"
                    f"   {chunk['metadata']['source']}"
                )
                seen.add(title)

        sources_text = "\n\n".join(source_lines) if source_lines else "No sources found"
        confidence_display = f"{confidence_label} ({confidence_score:.3f})"

        return answer, confidence_display, sources_text

    except Exception as e:
        return f"Error: {str(e)}", "Error", "No sources"

print("query_rag function ready!")

In [ ]:
demo = build_ui()

demo.launch(
    share=True,
    debug=False,
    quiet=False
)